In [ ]:
!pip install kagglehub librosa timm -q

In [ ]:
import kagglehub
path = kagglehub.dataset_download("soumendraprasad/sound-of-114-species-of-birds-till-2022")
print("Path to dataset files:", path)

In [ ]:
import os
from pathlib import Path
import numpy as np
import librosa
import torch
import torch.nn as nn
import torch.optim as optim
import timm
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from collections import Counter
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'models').exists():
    for parent in PROJECT_ROOT.parents:
        if (parent / 'models').exists() and (parent / 'notebooks').exists():
            PROJECT_ROOT = parent
            break

MODELS_DIR = PROJECT_ROOT / 'models' / 'audio'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = MODELS_DIR / 'bird_species_audio_classifier.pth'
print(f'Model path: {MODEL_PATH}')


In [ ]:
# scan dataset
audio_paths = []
labels = []

for root, dirs, files in os.walk(path):
    for f in files:
        if f.endswith(('.wav', '.mp3', '.ogg', '.flac')):
            label = os.path.basename(root)
            audio_paths.append(os.path.join(root, f))
            labels.append(label)

print(f"Total files: {len(audio_paths)}")
print(f"Unique species: {len(set(labels))}")

In [ ]:
le = LabelEncoder()
encoded_labels = le.fit_transform(labels)
NUM_CLASSES = len(le.classes_)
print(f"Classes: {NUM_CLASSES}")

In [ ]:
SR = 22050
DURATION = 5
N_MELS = 128
HOP_LENGTH = 512
N_FFT = 2048
MAX_LEN = 128

def extract_melspec(file_path):
    try:
        y, sr = librosa.load(file_path, sr=SR, duration=DURATION, mono=True)
        target_len = SR * DURATION
        if len(y) < target_len:
            y = np.pad(y, (0, target_len - len(y)))
        else:
            y = y[:target_len]
        mel = librosa.feature.melspectrogram(y=y, sr=SR, n_mels=N_MELS,
                                              hop_length=HOP_LENGTH, n_fft=N_FFT)
        mel_db = librosa.power_to_db(mel, ref=np.max)
        mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-6)
        if mel_db.shape[1] < MAX_LEN:
            mel_db = np.pad(mel_db, ((0, 0), (0, MAX_LEN - mel_db.shape[1])))
        else:
            mel_db = mel_db[:, :MAX_LEN]
        return mel_db.astype(np.float32)
    except:
        return None

In [ ]:
print("Extracting features (this takes a while)...")
features, valid_labels = [], []

for fp, lbl in tqdm(zip(audio_paths, encoded_labels), total=len(audio_paths)):
    mel = extract_melspec(fp)
    if mel is not None:
        features.append(mel)
        valid_labels.append(lbl)

X = np.array(features)[:, np.newaxis, :, :]   # (N, 1, 128, 128)
y = np.array(valid_labels)
print(f"Feature array: {X.shape}")

In [ ]:
# drop singleton classes
counts = Counter(y)
keep_mask = np.array([counts[lbl] >= 2 for lbl in y])
X, y = X[keep_mask], y[keep_mask]

unique_labels = np.unique(y)
remap = {old: new for new, old in enumerate(unique_labels)}
y = np.array([remap[lbl] for lbl in y])
NUM_CLASSES = len(unique_labels)
le.classes_ = le.classes_[unique_labels]
print(f"After filtering — samples: {len(y)}, classes: {NUM_CLASSES}")

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape}  Val: {X_val.shape}")

In [ ]:
class BirdDataset(Dataset):
    def __init__(self, X, y, augment=False):
        self.X = torch.tensor(X)
        self.y = torch.tensor(y, dtype=torch.long)
        self.augment = augment

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x = self.X[idx].clone()
        if self.augment:
            if np.random.rand() > 0.5:
                t = np.random.randint(0, 20)
                t0 = np.random.randint(0, MAX_LEN - t)
                x[:, :, t0:t0+t] = 0
            if np.random.rand() > 0.5:
                f = np.random.randint(0, 20)
                f0 = np.random.randint(0, N_MELS - f)
                x[:, f0:f0+f, :] = 0
        return x, self.y[idx]

train_ds = BirdDataset(X_train, y_train, augment=True)
val_ds   = BirdDataset(X_val,   y_val,   augment=False)

train_dl = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
class BirdCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.base = timm.create_model('efficientnet_b0', pretrained=True,
                                       in_chans=1, num_classes=num_classes)

    def forward(self, x):
        return self.base(x)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using: {device}")
model = BirdCNN(NUM_CLASSES).to(device)

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

backbone_params = [p for n, p in model.named_parameters() if 'classifier' not in n]
head_params     = [p for n, p in model.named_parameters() if 'classifier' in n]

optimizer = optim.AdamW([
    {'params': backbone_params, 'lr': 1e-4},
    {'params': head_params,     'lr': 3e-4}
], weight_decay=1e-4)

scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=40)

EPOCHS = 40
best_val_acc = 0

for epoch in range(EPOCHS):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
        correct += (out.argmax(1) == yb).sum().item()
        total += xb.size(0)

    scheduler.step()

    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for xb, yb in val_dl:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            val_correct += (out.argmax(1) == yb).sum().item()
            val_total += xb.size(0)

    train_acc = correct / total * 100
    val_acc   = val_correct / val_total * 100

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), MODEL_PATH)

    print(f"Epoch {epoch+1:02d}/{EPOCHS}  Loss: {total_loss/total:.4f}  "
          f"Train: {train_acc:.2f}%  Val: {val_acc:.2f}%"
          f"{'  * saved' if val_acc == best_val_acc else ''}")

print(f"\nBest Val Acc: {best_val_acc:.2f}%")
print(f"Model saved to: {MODEL_PATH}")

In [ ]:
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()

all_preds, all_true = [], []
with torch.no_grad():
    for xb, yb in val_dl:
        out = model(xb.to(device))
        all_preds.extend(out.argmax(1).cpu().numpy())
        all_true.extend(yb.numpy())

from sklearn.metrics import classification_report
present_labels = np.unique(all_true)
print(classification_report(all_true, all_preds,
                             labels=present_labels,
                             target_names=le.classes_[present_labels],
                             zero_division=0))

In [ ]:
def predict(file_path):
    mel = extract_melspec(file_path)
    if mel is None:
        return "Could not load file"
    x = torch.tensor(mel[np.newaxis, np.newaxis]).to(device)
    model.eval()
    with torch.no_grad():
        probs = torch.softmax(model(x), dim=1).cpu().numpy()[0]
    top5 = probs.argsort()[-5:][::-1]
    for i in top5:
        print(f"{le.classes_[i]:35s}  {probs[i]*100:.2f}%")

# predict("path/to/your/audio.wav")